# 05 · Train LoRA

QLoRA trên Qwen3.5-9B bằng Unsloth (`training/train_lora.py`): base 4-bit, LoRA r16 trên attention và MLP, loss chỉ trên phần trả lời.
Khởi tạo adapter mới từ model gốc với dữ liệu của `04_training_data`. Số dòng, số bước và thời gian được ghi theo lượt train thực tế.

**Độ dài dòng:** mỗi dòng dài từ 9k đến 19k token (trung vị khoảng 12,8k) vì prompt chứa rulebook và schema của shop. Loss chỉ tính trên phần trả lời
(khoảng 4% số token). `MAX_LEN = 16384` giữ khoảng 92% số dòng, `24576` giữ hết, dưới khoảng 9k thì không còn dòng nào.

**Phần cứng:**
- Công thức đầy đủ (`MAX_LEN = 32768`) đã chạy trên GPU bf16 24 GB (RTX PRO 4000 Blackwell, đỉnh 15,6 GiB, khoảng 80 giây mỗi bước 8 dòng).
- GPU 16 GB fp16 (T4, P100): chưa thử, có thể thử `MAX_LEN = 16384` với `LIMIT_ROWS` nhỏ để học, nhưng có thể hết bộ nhớ, và kernel fla có thể không chạy.
- Phiên Kaggle có giới hạn thời gian: checkpoint được lưu mỗi `SAVE_STEPS` bước vào `runs/` (Kaggle: `/kaggle/working`, giữ lại sau phiên),
  chạy lại notebook với `RESUME = True` (đính kèm output của phiên trước làm dataset rồi chép vào `OUT`) để train tiếp.

**Môi trường:** stack train (Unsloth, torch, peft, fla) khác môi trường suy luận. Máy local: `.venv-train`, Kaggle: notebook này tự cài
(dùng một phiên riêng, không chạy chung với 02 / 04).

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with your
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (if the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hana200804/datamate-agent-Locked-In1.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/datamate-agent-Locked-In1")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/datamate-agent-Locked-In1"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("train")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
SFT = "runs/sft/sql_rft_nb_notes.jsonl"   # from 04_training_data
INIT_ADAPTER = None                     # train a new adapter from the base model
OUT = "runs/lora/sql_rft_nb"
MAX_LEN = 32768                            # tokens per row, longer rows are dropped
LR = 1e-4                                  # fresh adapter
EPOCHS = 1
GRAD_ACCUM = 8
LIMIT_ROWS = None                          # e.g. 64 for a short learning run
MAX_STEPS = -1                             # e.g. 3 for a speed / memory check
SAVE_STEPS = 50
RESUME = True                              # continue from the newest checkpoint in OUT
COMPLETION_LOGITS = True                   # LM head only on reply positions: same loss, ~12% faster

In [ ]:
py = str(env.root / ".venv-train/bin/python") if (env.root / ".venv-train/bin/python").exists() else env.python   # the training stack
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
sft = Path(SFT) if Path(SFT).is_absolute() else env.root / SFT
assert sft.exists(), f"{sft} not found: build it in 04_training_data"
print("training python:", py, "| precision:", env.training_precision(), "| rows:", sum(1 for _ in open(sft, encoding="utf-8")))

## Kernel fla

24 trên 32 lớp của Qwen3.5-9B là Gated DeltaNet. Có `fla-core` thì dùng kernel Triton (nhanh khoảng 90 lần mỗi lớp). Không có thì transformers lặng lẽ
dùng bản PyTorch tham chiếu. Ô này so kernel với bản tham chiếu trên dữ liệu ngẫu nhiên. Trượt thì chạy không có fla (đúng nhưng chậm).

In [ ]:
extra_env = {}
if not env.can_run_llm:
    print("no bf16 GPU with >= 24 GB: kernel check skipped")
else:
    try:
        env.run([py, "training/check_fla.py", "--T", "2048"], show="fla check|T=|rel L2|Traceback|Error")
    except RuntimeError as e:
        print("fla unusable here, training without it:", e)
        extra_env = {"PYTHONPATH": str(env.root / "training/no_fla")}

In [ ]:
if env.can_run_llm:
    cmd = [py, "training/train_lora.py", sft, "--out", out, "--model", env.base_model(), "--max-len", MAX_LEN, "--r", 16, "--alpha", 32, "--lr", LR,
           "--epochs", EPOCHS, "--grad-accum", GRAD_ACCUM, "--end", "<|im_end|>", "--no-thinking", "--max-steps", MAX_STEPS, "--save-steps", SAVE_STEPS]
    cmd += (["--init-adapter", env.adapter(INIT_ADAPTER)] if INIT_ADAPTER else []) + (["--limit-rows", LIMIT_ROWS] if LIMIT_ROWS else [])
    cmd += (["--resume"] if RESUME else []) + (["--completion-logits"] if COMPLETION_LOGITS else [])
    env.run(cmd, log=out.with_name(out.name + ".log"), env=extra_env, show="examples|tokens:|precision|resuming|continuing|'loss'|steps in|Traceback|Error|OutOfMemory", every=120)
else:   # Kaggle T4 / P100 or a CPU session: the log of the same run made on a 24 GB GPU
    env.use_saved(out.name + ".log", out.with_name(out.name + ".log"))

## Loss

Loss theo bước (đọc từ `trainer_state.json` của checkpoint mới nhất, hoặc từ log).

In [ ]:
import ast
log_lines = out.with_name(out.name + ".log").read_text(encoding="utf-8", errors="replace").replace("\r", "\n").splitlines()
losses = [float(ast.literal_eval(l.strip())["loss"]) for l in log_lines if l.strip().startswith("{'loss'")]
if losses:
    s = pd.Series(losses)
    ax = s.plot(alpha=0.3, figsize=(7, 3), title="training loss")
    s.rolling(max(1, len(s) // 20)).mean().plot(ax=ax)
print(len(losses), "logged steps; adapter:", out / "adapter", "exists" if (out / "adapter/adapter_model.safetensors").exists() else "not here (saved log only)" if not env.can_run_llm else "MISSING")

## Dùng adapter vừa train

Trong `02_inference` đặt `LORA` bằng đường dẫn `OUT/adapter` (ví dụ `"runs/lora/sql_rft_nb/adapter"`), chạy trên holdout của train
(`SPLIT = "train"`, `IDS = "splits/holdout600.txt"`) hoặc public, rồi so trong `03_evaluation`. Holdout chỉ là smoke test: adapter đã học các
dòng của chính các câu đó (docs/leaderboard.md).